[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/41_tanh_solution.ipynb)

# Solution: Tanh, Backward & Soft-Capping

$$\text{tanh}(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

$$\frac{d}{dx}\text{tanh}(x) = 1 - \text{tanh}^2(x)$$

$$\text{soft\_cap}(x) = \text{cap} \cdot \text{tanh}\left(\frac{x}{\text{cap}}\right)$$

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def my_tanh(x: torch.Tensor) -> torch.Tensor:
    # Equivalent to (e^x - e^-x)/(e^x + e^-x), but numerically stable
    # Divide numerator & denominator by e^x  →  2·sigmoid(2x) - 1
    return 2.0 / (1.0 + torch.exp(-2.0 * x)) - 1.0


def tanh_backward(grad_output: torch.Tensor, tanh_output: torch.Tensor) -> torch.Tensor:
    return grad_output * (1 - tanh_output ** 2)


def soft_cap_logits(logits: torch.Tensor, cap: float) -> torch.Tensor:
    return cap * my_tanh(logits / cap)

In [ ]:
# Verify
x = torch.tensor([-2., -1., 0., 1., 2.])
print('my_tanh:', my_tanh(x))
print('ref:    ', torch.tanh(x))

t = my_tanh(x)
print('backward:', tanh_backward(torch.ones_like(t), t))

logits = torch.tensor([-50., 0., 50.])
print('soft_cap(cap=30):', soft_cap_logits(logits, 30.0))

# Run judge
from torch_judge import check
check('tanh')